# Hybrid Graph + Vector RAG [Step 5 - Structure and Content Together]

> **MLCourse - Agentic AI - Advanced RAG - Graph RAG**

The two retrievers we now have fail in exactly opposite ways:

| | vector search | graph traversal |
|---|---|---|
| good at | what a passage *says* | how entities *relate* |
| bad at | multi-hop, aggregation, relational | quoting, nuance, anything not in a triple |
| failure mode | returns plausible but non-answering text | returns nothing, or a true-but-useless fact |
| provenance | the chunk itself | edges (only if you kept `doc_id`) |

That complementarity is the whole argument for combining them - and this
notebook builds the combination, routes questions between them, and compares all
three approaches on a shared question set.

### 1. Setup


In [1]:
import os
import re
import time
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
from dotenv import load_dotenv


def find_env(start=None):
    """Walk up from the notebook directory until a .env file appears."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No .env found walking up from " + str(start))


ENV_PATH = find_env()
load_dotenv(ENV_PATH)
DATA_DIR = ENV_PATH.parent / "data"

print("env file :", ENV_PATH)
print("data dir :", DATA_DIR)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))

env file : D:\projects\python\MLCourse\03_agentic_ai\.env
data dir : D:\projects\python\MLCourse\03_agentic_ai\data
GROQ_API_KEY present: True


In [2]:
from langchain_groq import ChatGroq

GROQ_MODEL = "qwen/qwen3.8-27b"          # verified available on this account
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)


def clean(text):
    """Strip any <think>...</think> block a reasoning model may emit."""
    return THINK_RE.sub("", text).strip()


def ask(prompt, retries=4, pause=1.5):
    """Call Groq with exponential backoff. Free tier is roughly 8000 tokens/minute,
    so every loop in these notebooks paces itself and retries on rate limits."""
    delay = 5.0
    for attempt in range(retries):
        try:
            answer = clean(llm.invoke(prompt).content)
            time.sleep(pause)
            return answer
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print(f"  [retry {attempt + 1}] {type(exc).__name__} - sleeping {delay:.0f}s")
            time.sleep(delay)
            delay *= 2


print("Groq model:", GROQ_MODEL)
print("smoke test:", ask("Reply with exactly one word: ready"))

Groq model: qwen/qwen3.8-27b


smoke test: ready


In [3]:
ALICE_PATH = DATA_DIR / "alice.txt"
raw_text = ALICE_PATH.read_text(encoding="utf-8-sig")

paragraphs = [" ".join(p.split()) for p in raw_text.split("\n\n") if len(p.strip()) > 200]
print("paragraphs:", len(paragraphs))

paragraphs: 237


### Extracted with the LLM in notebook 02 and saved here so notebooks 03-05 do not


In [ ]:
# each re-pay the extraction cost. In a real system this is exactly what you do:
# extraction is a ONE-OFF indexing step whose output is persisted.
TRIPLES = [
    ("Alice", "follows", "White Rabbit"),
    ("Alice", "falls down", "rabbit hole"),
    ("White Rabbit", "carries", "pocket watch"),
    ("White Rabbit", "serves", "Duchess"),
    ("White Rabbit", "is herald for", "King of Hearts"),
    ("Alice", "drinks from", "little bottle"),
    ("little bottle", "causes", "shrinking"),
    ("Alice", "eats", "cake"),
    ("cake", "causes", "growing"),
    ("Alice", "meets", "Caterpillar"),
    ("Caterpillar", "sits on", "mushroom"),
    ("mushroom", "causes", "size change"),
    ("Caterpillar", "advises", "Alice"),
    ("Alice", "meets", "Cheshire Cat"),
    ("Cheshire Cat", "belongs to", "Duchess"),
    ("Cheshire Cat", "vanishes leaving", "grin"),
    ("Cheshire Cat", "directs Alice to", "Mad Hatter"),
    ("Alice", "attends", "mad tea party"),
    ("Mad Hatter", "attends", "mad tea party"),
    ("March Hare", "attends", "mad tea party"),
    ("Dormouse", "attends", "mad tea party"),
    ("Mad Hatter", "quarrelled with", "Time"),
    ("Duchess", "nurses", "baby"),
    ("baby", "turns into", "pig"),
    ("Duchess", "employs", "Cook"),
    ("Cook", "throws", "pepper"),
    ("Alice", "meets", "Queen of Hearts"),
    ("Queen of Hearts", "orders", "beheadings"),
    ("Queen of Hearts", "plays", "croquet"),
    ("croquet", "uses", "flamingo"),
    ("croquet", "uses", "hedgehog"),
    ("Queen of Hearts", "is married to", "King of Hearts"),
    ("Queen of Hearts", "commands", "playing cards"),
    ("playing cards", "paint", "white roses"),
    ("Knave of Hearts", "is accused of stealing", "tarts"),
    ("Queen of Hearts", "baked", "tarts"),
    ("King of Hearts", "presides over", "trial"),
    ("Knave of Hearts", "stands at", "trial"),
    ("Mad Hatter", "testifies at", "trial"),
    ("Alice", "testifies at", "trial"),
    ("Gryphon", "takes Alice to", "Mock Turtle"),
    ("Queen of Hearts", "sends", "Gryphon"),
    ("Mock Turtle", "tells", "his history"),
    ("Mock Turtle", "dances", "Lobster Quadrille"),
    ("Gryphon", "dances", "Lobster Quadrille"),
]

print(len(TRIPLES), "curated (subject, relation, object) triples")


In [5]:
import networkx as nx
from sentence_transformers import SentenceTransformer
import numpy as np

# --- the graph side --------------------------------------------------------
G = nx.MultiDiGraph()
for s, r, o in TRIPLES:
    G.add_edge(s, o, relation=r)
NODES = sorted(G.nodes())

# --- the vector side -------------------------------------------------------
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_vectors = encoder.encode(paragraphs, normalize_embeddings=True,
                             batch_size=64, show_progress_bar=False)
node_vectors = encoder.encode(NODES, normalize_embeddings=True)

print(f"graph : {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"vector: {doc_vectors.shape[0]} paragraphs")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

graph : 38 nodes, 45 edges
vector: 237 paragraphs


In [6]:
def vector_context(question, top_k=3):
    sims = doc_vectors @ encoder.encode([question], normalize_embeddings=True)[0]
    ids = [int(i) for i in np.argsort(sims)[::-1][:top_k]]
    return ids, "\n\n".join(f"[doc_{i}] {paragraphs[i]}" for i in ids)


def link_entities(question, threshold=0.42, top_n=4):
    literal = [n for n in NODES if n.lower() in question.lower()]
    sims = node_vectors @ encoder.encode([question], normalize_embeddings=True)[0]
    fuzzy = [NODES[i] for i in np.argsort(sims)[::-1][:top_n] if sims[i] >= threshold]
    return list(dict.fromkeys(literal + fuzzy))


def graph_context(question, hops=2, limit=45):
    seeds = link_entities(question)
    und = G.to_undirected(as_view=False)
    keep = set()
    for seed in seeds:
        if seed in und:
            keep |= set(nx.ego_graph(und, seed, radius=hops).nodes())
    sub = G.subgraph(keep)
    lines = sorted({f"{u} {d['relation']} {v}." for u, v, d in sub.edges(data=True)})
    return seeds, "\n".join(lines[:limit])


print("both retrievers ready")

both retrievers ready


### 2. The simple combination: give the LLM both

The cheapest hybrid puts both context types in one prompt under clear headings.
The headings matter - the model needs to know that one block is verbatim source
text (quotable) and the other is derived structure (reliable for relations, not
for wording).

In [7]:
HYBRID_PROMPT = (
    "You answer questions about 'Alice's Adventures in Wonderland' using two "
    "kinds of evidence.\n\n"
    "STRUCTURED FACTS (from a knowledge graph - reliable for who/what relates to "
    "whom, but contains no wording from the book):\n{facts}\n\n"
    "SOURCE PASSAGES (verbatim from the book - quote these for anything about "
    "what was said or how it happened):\n{passages}\n\n"
    "Answer the question. Use the structured facts to establish relationships and "
    "the passages to quote specifics. If neither contains the answer, say so.\n\n"
    "Question: {question}\nAnswer:"
)


def hybrid_rag(question, hops=2, top_k=3):
    seeds, facts = graph_context(question, hops=hops)
    ids, passages = vector_context(question, top_k=top_k)
    answer = ask(HYBRID_PROMPT.format(facts=facts, passages=passages,
                                      question=question))
    return {"entities": seeds, "docs": ids, "answer": answer}


result = hybrid_rag("Who owns the cat that directed Alice to the Hatter, "
                    "and what did the cat look like when it vanished?")
print("linked entities:", result["entities"])
print("retrieved docs :", result["docs"])
print("\nanswer:")
print(result["answer"])

linked entities: ['Alice', 'Cheshire Cat']
retrieved docs : [124, 155, 56]

answer:
Based on the structured facts, the **Cheshire Cat** is the one who directed Alice to the Mad Hatter, and the **Duchess** is the owner of the Cheshire Cat.

Regarding the appearance of the cat when it vanished, the structured facts state that the "Cheshire Cat vanishes leaving grin," but they do not describe the visual details of the vanishing process. The provided source passages do not contain the specific scene where the Cheshire Cat vanishes (they discuss Alice walking toward the March Hare's house, the Cat sitting on a branch, and Alice talking about her cat Dinah). Therefore, while we know the cat left a grin, the specific visual description of the vanishing act is not present in the provided evidence.

**Answer:**
The cat that directed Alice to the Hatter is owned by the **Duchess**. According to the structured facts, the cat vanished leaving a **grin**, but the provided source passages do not con

That question was deliberately built to need both halves: *who owns the cat* is a
two-hop graph walk, *what it looked like when it vanished* is prose that exists
only in the text. Neither retriever alone can answer it.

### 3. Three approaches, one comparison

Now be systematic. Same questions, three retrievers, and an honest look at where
each one wins.

In [8]:
def graph_only(question, hops=2):
    seeds, facts = graph_context(question, hops=hops)
    return ask(
        "Answer using ONLY these knowledge-graph facts. If they are "
        "insufficient, say exactly what is missing.\n\n"
        f"Facts:\n{facts}\n\nQuestion: {question}\nAnswer:"
    )


def vector_only(question, top_k=3):
    _, passages = vector_context(question, top_k=top_k)
    return ask(
        "Answer using ONLY these passages. If they are insufficient, say "
        "exactly what is missing.\n\n"
        f"Passages:\n{passages}\n\nQuestion: {question}\nAnswer:"
    )


TEST_QUESTIONS = [
    ("relational", "What connects the Queen of Hearts to the Mock Turtle?"),
    ("content", "What did the Caterpillar say to Alice about keeping her temper?"),
]

for kind, question in TEST_QUESTIONS:
    print("=" * 78)
    print(f"[{kind}] {question}")
    print("=" * 78)
    print("\n-- GRAPH ONLY --")
    print(graph_only(question))
    time.sleep(3.0)
    print("\n-- VECTOR ONLY --")
    print(vector_only(question))
    time.sleep(3.0)
    print()

[relational] What connects the Queen of Hearts to the Mock Turtle?

-- GRAPH ONLY --


The provided facts do not contain a direct connection between the Queen of Hearts and the Mock Turtle.

Missing information:
1. A fact stating that the Queen of Hearts interacts with, commands, or is related to the Mock Turtle.
2. A fact linking the Queen of Hearts to the Gryphon (other than "Queen of Hearts sends Gryphon") and a fact linking the Gryphon to the Mock Turtle (other than "Gryphon takes Alice to Mock Turtle") in a way that establishes a direct connection between the Queen and the Mock Turtle, or a shared attribute/event involving both.

While the facts state "Queen of Hearts sends Gryphon" and "Gryphon takes Alice to Mock Turtle," there is no explicit fact connecting the Queen of Hearts directly to the Mock Turtle.



-- VECTOR ONLY --


The provided passages are insufficient to answer the question. They do not contain any mention of the Queen of Hearts or any connection between her and the Mock Turtle.



[content] What did the Caterpillar say to Alice about keeping her temper?

-- GRAPH ONLY --


The provided facts state that "Caterpillar advises Alice," but they do not contain the specific content of that advice or any quote regarding keeping her temper.

Missing: The specific advice or quote given by the Caterpillar to Alice about keeping her temper.



-- VECTOR ONLY --


The provided passages do not contain any information about the Caterpillar telling Alice to keep her temper. The passages only mention the Caterpillar's short remarks, its instruction to Alice to tell it who it is, and its final remark about the mushroom making her grow taller or shorter.


In [9]:
for kind, question in TEST_QUESTIONS:
    print("=" * 78)
    print(f"[{kind}] {question}")
    print("=" * 78)
    print(hybrid_rag(question)["answer"])
    print()
    time.sleep(3.0)

[relational] What connects the Queen of Hearts to the Mock Turtle?


Based on the provided evidence, there is **no direct connection** stated between the Queen of Hearts and the Mock Turtle.

**Analysis of Evidence:**

1.  **Structured Facts:**
    *   The facts list the Queen of Hearts' relationships and actions: she is married to the King of Hearts, commands playing cards, bakes tarts, plays croquet, orders beheadings, and **sends the Gryphon**.
    *   The facts list the Mock Turtle's relationships and actions: he dances the Lobster Quadrille, tells his history, and is taken to Alice by the **Gryphon**.
    *   While the **Gryphon** acts as a link (the Queen sends the Gryphon, and the Gryphon takes Alice to the Mock Turtle), the structured facts do not explicitly state a relationship between the Queen and the Mock Turtle themselves.

2.  **Source Passages:**
    *   The provided passages (doc_178, doc_180, doc_181) focus exclusively on the Mock Turtle's sobbing, his school history in the sea, and his subjects (Mystery, Seaography, Drawling, etc.).
  

[content] What did the Caterpillar say to Alice about keeping her temper?


The provided structured facts and source passages do not contain the answer to this question.

The structured facts confirm that "Caterpillar advises Alice" and "Caterpillar sits on mushroom," but they do not specify the content of the advice. The source passages provided ([doc_91], [doc_93], [doc_134]) discuss Alice's irritation with the Caterpillar's short remarks, the Caterpillar's departure, and a separate incident at the Mad Tea Party, but none of them include the specific quote where the Caterpillar tells Alice to "Keep your temper."



Read those six answers as a grid. The expected pattern - and the reason hybrid is
the right default:

- On the **relational** question, graph-only is precise and vector-only waffles.
- On the **content** question, vector-only quotes the text and graph-only can
  only say that advice was given.
- Hybrid is at least as good as the better of the two in both cases, at the cost
  of a longer prompt.

### 4. Routing: pick the retriever per question

Feeding both contexts every time is simple but wasteful - you pay for graph
verbalisation on pure content questions and for passages on pure relational
ones. A **router** classifies the question first.

This is the same idea as [`../06_adaptive_rag`](../06_adaptive_rag/README.md),
applied to the graph/vector choice.

In [10]:
ROUTER_PROMPT = (
    "Classify this question into exactly one category and reply with the "
    "category word only.\n\n"
    "RELATIONAL - about how entities connect, multi-hop chains, or which "
    "entities share a property (needs a knowledge graph)\n"
    "CONTENT - about what a passage says, quotes, descriptions, or wording "
    "(needs the source text)\n"
    "BOTH - needs relationships AND specific wording\n\n"
    "Question: {question}\nCategory:"
)


def routed_rag(question):
    route = ask(ROUTER_PROMPT.format(question=question)).strip().upper()
    route = next((r for r in ["RELATIONAL", "CONTENT", "BOTH"] if r in route), "BOTH")
    if route == "RELATIONAL":
        return route, graph_only(question)
    if route == "CONTENT":
        return route, vector_only(question)
    return route, hybrid_rag(question)["answer"]


ROUTING_TESTS = [
    "Which characters attend both the tea party and the trial?",
    "How does the Cheshire Cat's grin behave when it vanishes?",
    "Who sent the Gryphon to Alice, and what did the Gryphon do next?",
]

for question in ROUTING_TESTS:
    route, answer = routed_rag(question)
    print(f"[{route}] {question}")
    print("   ", answer.replace("\n", " ")[:340])
    print()
    time.sleep(3.0)

[RELATIONAL] Which characters attend both the tea party and the trial?
    Based on the provided facts, the characters who attend the mad tea party are: 1. Alice 2. Dormouse 3. Mad Hatter 4. March Hare  The characters who attend the trial (explicitly stated as "testifies at trial" or "stands at trial") are: 1. Alice 2. Knave of Hearts 3. Mad Hatter  The characters present in both lists are **Alice** and **Mad Ha



[CONTENT] How does the Cheshire Cat's grin behave when it vanishes?
    The provided passages are insufficient to answer the question. They mention the Cheshire Cat's grin appearing (doc_154) and its head fading away (doc_164), but they do not describe the specific behavior of the grin itself when it vanishes.



[BOTH] Who sent the Gryphon to Alice, and what did the Gryphon do next?
    Based on the structured facts, the **Queen of Hearts** is the one who sent the Gryphon to Alice.  According to the source passage [doc_175], after the Queen ordered the Gryphon to take Alice to see the Mock Turtle, the Gryphon's next action was to **wait** with Alice. The text states that Alice "waited" because she thought it would be saf



### 5. Cost accounting

Be explicit about what each approach costs, because hybrid is not free.

| approach | index-time cost | query-time cost | prompt size |
|---|---|---|---|
| vector only | embed each chunk (cheap, no LLM) | 1 LLM call | k chunks |
| graph only | **1 LLM call per chunk** to extract | 1-2 LLM calls (linking + answer) | verbalised subgraph |
| hybrid | both of the above | 1-2 LLM calls | chunks **plus** facts |
| routed | both | 2 LLM calls (route + answer) | whichever one is needed |

The dominant term is graph **extraction**: one LLM call per chunk, paid once. For
a 10,000-chunk corpus that is a real budget item, and it is why Graph RAG is
usually deployed on a curated subset of documents rather than everything.

Routing pays for itself when the question mix is skewed - most production
corpora get far more content questions than relational ones.

### 6. Design guidance

**Use vector only** when your questions are "what does the documentation say
about X". This is most RAG, and adding a graph would be complexity for nothing.

**Add a graph** when you have a genuine relational domain - org charts, supply
chains, code dependencies, medical ontologies, incident causality - and when
users ask multi-hop or aggregation questions.

**Always keep provenance.** Store `doc_id` on every edge (as in
[`02_entity_extraction.ipynb`](02_entity_extraction.ipynb)) so a graph-derived
claim can still be traced to source text. Without it, hybrid answers become
uncitable.

**Compose with the rest of the track.** The graph does not replace anything you
have already built:

```
   question
     |
     +-- route (graph / vector / both)          <- ../06_adaptive_rag
     |
     +-- vector path: transform -> hybrid retrieve -> rerank
     |      ../12_query_transformation, ../01_hybrid_search, ../11_reranking
     |
     +-- graph path: link entities -> traverse -> verbalise
     |
     +-- generate from combined context
```

### 7. Have the model summarise the tradeoff


In [11]:
print(ask(
    "You are a RAG architect. In 6-8 sentences, advise an engineer who has a "
    "working vector RAG system on a 50,000-document corpus and is considering "
    "adding a knowledge graph. Cover: the specific question types that justify "
    "it, the one-off extraction cost of one LLM call per chunk, entity "
    "resolution risk, and why a hybrid or routed design beats replacing vector "
    "search outright. Be concrete and do not oversell graphs."
))

A knowledge graph is only justified if your users frequently ask multi-hop relational questions, such as "Which suppliers for Product A also provide Component B?" or "What are the downstream impacts of this specific policy change?" that vector similarity alone cannot resolve. The primary cost is the one-off extraction phase, where you must run an LLM call on every one of your 50,000 chunks to identify entities and relationships, a process that is computationally expensive and requires careful prompt engineering to ensure consistency. You must also budget significant time for entity resolution, as the LLM will likely generate inconsistent labels (e.g., "IBM" vs. "International Business Machines") that require complex deduplication logic to merge into a coherent graph. Do not replace your existing vector search, as it remains superior for semantic similarity and unstructured fact retrieval; instead, implement a hybrid architecture where the graph handles structured, relational queries wh

### 8. Key takeaways

- Vector and graph retrieval fail in **opposite** directions - content versus
  structure - which is exactly why they combine well.
- The simple hybrid gives the LLM both context blocks under clear headings, so it
  knows which is quotable.
- A **router** avoids paying for both on every query and is the production shape.
- The dominant cost is **graph extraction at index time**; budget it, and
  consider building the graph over a curated subset.
- Keep `doc_id` provenance on edges so graph-derived claims stay citable.

That closes the Graph RAG module, and the retrieval-technique arc of this track.
Next: [`../10_rag_evaluation`](../10_rag_evaluation/README.md) - how to know
whether any of it is actually working.